Goals of the notebook
- create pseudobulk objects

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path 

import scanpy as sc
import pandas as pd 
import numpy as np 
import os
import anndata as ad
from dotenv import dotenv_values 

from hm3d_utils.utils import create_pseudobulks_for_multiple_levels, write_clean_h5ad

from project_utils import FileTracker

# Setup

In [3]:
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OBJECTS_DIR = DATADIR / "9_objects" 
OUTDIR = OBJECTS_DIR / "pseudobulks"

INPUT_FILE = OBJECTS_DIR / "singlecell" / "hypomap3d_singlecell_all.h5ad"

METADATA_FILE = OUTDIR / "hypomap3d_pseudobulk_all_metadata.csv"

CLUSTERING_COLS = [f"L{i}_ID" for i in range(2,6)]

OUTPUT_FILE = OUTDIR / f"hypomap3d_pseudobulk_all.h5ad"

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

# Load objects and insert pseudobulk specific metadata

In [4]:
adata = sc.read_h5ad(INPUT_FILE)

In [5]:
pb_md = pd.read_csv(METADATA_FILE, )

In [6]:
def build_id_mapping(n_df, id_pattern, target_pattern, levels=range(1, 6),):
    mapping = {}

    # --- 1. from wide dataframe ---
    for i in levels:
        id_col = id_pattern.format(i=i)
        target_col = target_pattern.format(i=i)

        if id_col in n_df.columns and target_col in n_df.columns:
            mapping.update(
                n_df[[id_col, target_col]]
                .dropna()
                .drop_duplicates()
                .set_index(id_col)[target_col]
                .to_dict()
            )

    return mapping

In [7]:
abclike_mapper = build_id_mapping(adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_abc_like")

descr_mapper = build_id_mapping(adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_descr")

In [8]:
adata.obs = adata.obs.loc[:, ["CellID", "Sample", ] + CLUSTERING_COLS]
# Merge with pb_md on "Sample", align using original index
adata.obs = adata.obs.merge(pb_md, on="Sample", how="left").set_index(adata.obs.index)

# Let's get a pseudobulk object for all the data

In [9]:
pb = create_pseudobulks_for_multiple_levels(
    adata,
    levels=CLUSTERING_COLS,
    sample_key="Sample",
    obs_to_keep=pb_md.columns
)

Processing clustering level: L2_ID
Processing clustering level: L3_ID
Processing clustering level: L4_ID
Processing clustering level: L5_ID


In [10]:
pb.obs["cell_type_descr"] = pb.obs.cell_type.map(descr_mapper)
pb.obs["cell_type_abc_like"] = pb.obs.cell_type.map(abclike_mapper)
pb.obs = pb.obs.rename(columns={"cell_type": "cell_type_ID"})
pb.obs.clustering_resolution = pb.obs.clustering_resolution.apply(lambda x: x.split("_")[0])

In [11]:
pb

AnnData object with n_obs × n_vars = 309480 × 56939
    obs: 'Sample', 'cell_type_ID', 'n_cells', 'library_size', 'Dataset', 'Author', 'Age', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'IsControl', 'Mean_age_in_weeks', 'sex_per_sample_inferred', 'Diet_loose', 'IsStrictControl', 'Diet_strict', 'lactation', 'Circadian_sac', 'Mean_age_in_months', 'Age_cat', 'Age_cat_ctrl', 'Strain_harmonized', 'clustering_resolution', 'cell_type_descr', 'cell_type_abc_like'

In [12]:
# reorder columns
pb.obs = pb.obs.loc[:, ['Sample','Dataset',
       'Author', 'Age', 'Suspension', 'Technology', 'Author_Sex_harmonized',
       'Strain', 'Diet', 'Treatment', 'IsControl', 'Mean_age_in_weeks',
       'sex_per_sample_inferred', 'Diet_loose', 'IsStrictControl',
       'Diet_strict', 'lactation', 'Circadian_sac', 'Mean_age_in_months',
       'Age_cat', 'Age_cat_ctrl', 'Strain_harmonized', 'clustering_resolution',
       'cell_type_ID', 'cell_type_descr', 'cell_type_abc_like',
       'n_cells', 'library_size', ]]

In [13]:
pb.obs.head(3)

,Sample,Dataset,Author,Age,Suspension,Technology,Author_Sex_harmonized,Strain,Diet,Treatment,...,Mean_age_in_months,Age_cat,Age_cat_ctrl,Strain_harmonized,clustering_resolution,cell_type_ID,cell_type_descr,cell_type_abc_like,n_cells,library_size
L2_ID.Gl2-35.SC1-01_SAMN18754353,SC1-01_SAMN18754353,SC1-01,Affinati,NaN,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,...,NaN,adult,adult,C57BL/6J,L2,Gl2-35,Gl2-35 VMH Nr5a1.Acvr1c,Gl2-35 VMH Nr5a1 Acvr1c Glut,399,6670056.0
L2_ID.Gl2-35.SC1-01_SAMN18754354,SC1-01_SAMN18754354,SC1-01,Affinati,NaN,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,...,NaN,adult,adult,C57BL/6J,L2,Gl2-35,Gl2-35 VMH Nr5a1.Acvr1c,Gl2-35 VMH Nr5a1 Acvr1c Glut,312,6260490.0
L2_ID.Gl2-35.SC1-01_SAMN18754356,SC1-01_SAMN18754356,SC1-01,Affinati,NaN,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,...,NaN,adult,adult,C57BL/6J,L2,Gl2-35,Gl2-35 VMH Nr5a1.Acvr1c,Gl2-35 VMH Nr5a1 Acvr1c Glut,244,3801971.0


In [14]:
with tracker.register(OUTPUT_FILE) as tf:
    tf.set_description('Full pseudobulked Hypomap3D (C + NC).')
    write_clean_h5ad(pb, tf.filepath)

... storing 'Sample' as categorical
... storing 'Dataset' as categorical
... storing 'Author' as categorical
... storing 'Age' as categorical
... storing 'Suspension' as categorical
... storing 'Technology' as categorical
... storing 'Author_Sex_harmonized' as categorical
... storing 'Strain' as categorical
... storing 'Diet' as categorical
... storing 'Treatment' as categorical
... storing 'sex_per_sample_inferred' as categorical
... storing 'Diet_loose' as categorical
... storing 'Diet_strict' as categorical
... storing 'lactation' as categorical
... storing 'Circadian_sac' as categorical
... storing 'Age_cat' as categorical
... storing 'Age_cat_ctrl' as categorical
... storing 'Strain_harmonized' as categorical
... storing 'clustering_resolution' as categorical
... storing 'cell_type_ID' as categorical
... storing 'cell_type_descr' as categorical
... storing 'cell_type_abc_like' as categorical


# Let's save smaller versions

### .. one where samples consisting of less than 5 cells are removed

In [15]:
pb_sub = pb[pb.obs.n_cells >= 5].copy()
pb_sub

AnnData object with n_obs × n_vars = 173827 × 56939
    obs: 'Sample', 'Dataset', 'Author', 'Age', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'IsControl', 'Mean_age_in_weeks', 'sex_per_sample_inferred', 'Diet_loose', 'IsStrictControl', 'Diet_strict', 'lactation', 'Circadian_sac', 'Mean_age_in_months', 'Age_cat', 'Age_cat_ctrl', 'Strain_harmonized', 'clustering_resolution', 'cell_type_ID', 'cell_type_descr', 'cell_type_abc_like', 'n_cells', 'library_size'

In [16]:
sub_fp = OUTDIR / f"hypomap3d_pseudobulk_all_min_5cells.h5ad"

with tracker.register(sub_fp) as tf:
    tf.set_description('Pseudobulked Hypomap3D (C + NC) with samples consisting of >= 5 cells.')
    write_clean_h5ad(pb_sub, tf.filepath)


### .. and then only the samples for a specific clustering resolution from those

In [17]:
for i in range(2, 6):
    subsub = pb_sub[pb_sub.obs.clustering_resolution == f"L{i}"].copy()

    subsub_fp = OUTDIR / f"hypomap3d_pseudobulk_all_min_5cells_L{i}.h5ad"
    with tracker.register(subsub_fp) as tf:
        tf.set_description(f'L{i} pseudobulked Hypomap3D (C + NC) with samples consisting of >= 5 cells.')
        write_clean_h5ad(subsub, tf.filepath)
